# S6E10 | Meta-Stack: Public OOF Pool + 17 New Members | LB 0.96185

One logistic regression over the logits of every published out-of-fold (OOF) prediction for this episode, plus
**17 members of mine that were not public before**: the models from ten rounds of ablation behind
[S6E10 | Encodings + CatBoost + RealMLP](https://www.kaggle.com/code/kagankoral/s6e10-encodings-catboost-realmlp-lb-0-961),
now in the dataset [s6e10-ablation-members-oof](https://www.kaggle.com/datasets/kagankoral/s6e10-ablation-members-oof).

- **Most of this notebook's value is other people's models.** The credits at the end list every source; please
  upvote them.
- **What is new here** is the 17-member dataset, and the measurement of what it adds on top of the full public pool.
- **Validation is nested:** the stacker is scored only on rows its weights never saw (meta folds
  `StratifiedKFold(5, shuffle=True, random_state=7)`, a different split from the members' own folds).
- **`submission.csv` blends in no submission files.** A test-only file has no OOF, so its weight could only be tuned
  on the leaderboard; everything in it is chosen by cross-validation. Section 4 writes a separate, clearly labelled
  hedge (`submission_twin.csv`) with a fixed weight.

**Result:** nested CV **0.962224** with 244 members, public LB **0.96185**. The public pool alone scores 0.962205; the
17 new members add **+0.000020, on 5 of 5 folds**, and two of them are among the most heavily weighted members of
the whole stack (section 3).

**Inputs:** the competition data and the datasets / notebook outputs listed in section 1. Every source is optional:
one that is missing or has changed is skipped and reported, and the stack uses the rest. CPU only, about 12 minutes.

## 1. Load the members

Each member becomes a column of log-odds. Columns already published as logits (outside [0, 1]) pass through
unchanged. Rows are matched to `train.csv` / `test.csv` by `id` wherever a file carries ids, and asserted to be in
that order otherwise.

| prefix | source | attach |
|---|---|---|
| *(none)* | Chris Deotte, 124 public OOF collection | dataset `cdeotte/s6e10-124-public-oof-and-test-predictions` |
| `tfm_` | Chris Deotte, tabular foundation models | dataset `cdeotte/s6e10-tfm-oof-and-test-predictions` |
| `gl2_` | dariushafshar, golem library (3 not in the 124) | dataset `dariushafshar/s6e10-golem-oof-library` |
| `st_` | sadamtorres, artifacts | dataset `sadamtorres/ps-s6e10-airline-satisfaction-artifacts` |
| `am_` | amanatar, forge pairs | dataset `amanatar/s6e10-forge-own-pairs` |
| `ab_` | abdullahsafwan333, 15-fold TabPFN-3.5 and GBDTs | dataset `abdullahsafwan333/oof-dataset` |
| `km_` | koumeimaki, route TFM / NN | dataset `koumeimaki/s6e10-route-tfm-nn-oof` |
| `s7_` | sachith7 (2 not in the 124) | dataset `sachith7/s6e10-stack-oof-predictions` |
| `kr_` | kratosyan, route-ID teacher | notebook `kratosyan/s6e10-xgboost-realmlp-route-id-teacher` |
| `kg_` | my Encodings + CatBoost + RealMLP notebook | notebook `s6e10-encodings-catboost-realmlp-lb-0-961` (by this notebook's author) |
| `yk_` | Vladimir Demidov, RealMLP | notebook `yekenot/ps-s6-e10-realmlp-pytabkit` |
| `hm_` | Hermengardo, CatBoost | notebook `hermengardo/single-catboost-eda-ps6e10` |
| `dk_` | dmitriykolmikov, all-categorical CatBoost | dataset `dmitriykolmikov/s6e10-catboost-all-categorical-oof-and-test` |
| `av5_` | amanatar, original-prior stack v5 (base members only) | notebook `amanatar/s6e10-the-original-prior-stack-v5` |
| `gpx_` | goodpjw2008, 17 extra members | dataset `goodpjw2008/s6e10-goodpjw2008-extra-members-oof` |
| **`kx_`** | **17 new members (this notebook's contribution)** | dataset `s6e10-ablation-members-oof` (by this notebook's author) |
| `cx_` | candidate members (all-categorical CatBoosts, TabPFN-3.5), screened in section 2b | the candidate notebooks' outputs (by this notebook's author) |

Blend- and stack-level OOFs from other sources are left out on purpose: stacking a stack double-counts its members.
One is inside the 124-member collection (najiama's blend, the best single column at 0.961744); it is kept, as in
the other public stacks built on that collection.

In [ ]:
import glob, os, time, warnings
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings('ignore')

T0 = time.time()
INPUT = '/kaggle/input'


def find(pattern, hint):
    hits = sorted(p for p in glob.glob(f'{INPUT}/**/{pattern}', recursive=True) if hint in p)
    assert hits, f'not attached: {pattern} ({hint})'
    return hits


tr = pd.read_csv(find('train.csv', 'playground-series-s6e10')[0])
te = pd.read_csv(find('test.csv', 'playground-series-s6e10')[0])
y = tr['satisfaction'].astype(int).values


def logit(p):
    p = np.clip(p, 1e-15, 1 - 1e-15)
    return np.clip(np.log(p / (1 - p)), -30, 30)


names, cols, tcols, source = [], [], [], []
OURS = 'ablation members (new)'
CAND = 'candidates (new)'


def add(name, src, o, t):
    o, t = np.asarray(o, dtype=np.float64).ravel(), np.asarray(t, dtype=np.float64).ravel()
    assert len(o) == len(y) and len(t) == len(te), (name, len(o), len(t))
    assert np.isfinite(o).all() and np.isfinite(t).all(), name
    already_logit = o.min() < 0 or o.max() > 1          # published as logits: pass through
    names.append(name); source.append(src)
    cols.append(o if already_logit else logit(o)); tcols.append(t if already_logit else logit(t))


def by_id(path, ids, col):
    return pd.read_csv(path).set_index('id').loc[ids, col].values

In [ ]:
def l_cdeotte124():
    d = os.path.dirname(find('oof_gp_cat_ctr.npy', '124-public')[0])
    for f in sorted(glob.glob(f'{d}/oof_*.npy')):
        m = os.path.basename(f)[4:-4]
        add(m, 'cdeotte 124', np.load(f), np.load(f'{d}/test_{m}.npy'))

def l_cdeotte_tfm():
    d = os.path.dirname(find('oof_tabfm.npy', 'tfm-oof')[0])
    for f in sorted(glob.glob(f'{d}/oof_*.npy')):
        m = os.path.basename(f)[4:-4]
        add('tfm_' + m, 'cdeotte TFM', np.load(f), np.load(f'{d}/test_{m}.npy'))

def l_golem():
    d = os.path.dirname(find('oof_o_tabpfn35.npy', 'golem')[0])
    for f in sorted(glob.glob(f'{d}/oof_*.npy')):
        m = os.path.basename(f)[4:-4]
        if len(m) > 1:                    # a ... n are in the 124; o_tabpfn35 onwards were added later, new ones too
            add('gl2_' + m, 'dariushafshar golem', np.load(f), np.load(f'{d}/test_{m}.npy'))

def l_sadamtorres():
    d = os.path.dirname(find('preds_e1_allcat_cat.npz', 'artifacts')[0])
    for f in sorted(glob.glob(f'{d}/preds_*.npz')) + sorted(glob.glob(f'{d}/orig_*_pred.npz')):
        z = np.load(f)
        if 'oof' in z and 'test' in z and len(z['oof']) == len(y) and np.isfinite(z['oof']).all():
            add('st_' + os.path.basename(f)[:-4].replace('preds_', ''), 'sadamtorres', z['oof'], z['test'])

def l_forge():
    d = os.path.dirname(find('oof_forge_cat.npy', 'forge')[0])
    for f in sorted(glob.glob(f'{d}/oof_forge_*.npy')):
        m = os.path.basename(f)[4:-4]
        add('am_' + m, 'amanatar forge', np.load(f), np.load(f'{d}/test_{m}.npy'))

def l_abdullah():
    d = os.path.dirname(find('tabpfn_catfd10_oof.csv', 'oof-dataset')[0])
    for m in ('cat_v3', 'tabpfn_catfd10', 'tabpfn_raw', 'tabpfn_te_cnt', 'xgb_v3'):
        add('ab_' + m, 'abdullahsafwan333', by_id(f'{d}/{m}_oof.csv', tr['id'], 'pred'),
            by_id(f'{d}/{m}_test.csv', te['id'], 'pred'))

def l_koumeimaki():
    d = os.path.dirname(find('train_ids_folds.csv', 'route-tfm-nn-oof')[0])
    assert (pd.read_csv(f'{d}/train_ids_folds.csv')['id'].values == tr['id'].values).all()
    assert (pd.read_csv(f'{d}/test_ids.csv')['id'].values == te['id'].values).all()
    for m in ('cat_w1', 'lgb_w1', 'rmlp_f_nc', 'tabicl_fdcat_c250k', 'tabm_bcd_rc_v2', 'tabpfn_catfd10', 'xgb_w1'):
        add('km_' + m, 'koumeimaki', np.load(f'{d}/oof_{m}.npy'), np.load(f'{d}/test_{m}.npy'))

def l_sachith7():
    d = os.path.dirname(find('lgb_cpu_strong.npz', 'stack-oof-predictions')[0])
    for m in ('lgb_cpu_strong', 'xgb_A_auxall'):                           # the other 22 are in the 124
        z = np.load(f'{d}/{m}.npz')
        add('s7_' + m, 'sachith7', z['oof'], z['test'])

def l_kratosyan():
    for m in ('rmlp_route_ogaux', 'xgb_route_ogaux'):
        z = np.load(find(f'{m}.npz', 'route-id-teacher')[0])
        add('kr_' + m, 'kratosyan', z['oof'], z['test'])

def l_encodings_notebook():
    d = os.path.dirname(find('oof_members.csv', 'encodings-catboost-realmlp')[0])
    o = pd.read_csv(f'{d}/oof_members.csv').set_index('id').loc[tr['id']]
    t = pd.read_csv(f'{d}/test_members.csv').set_index('id').loc[te['id']]
    for m in ('lgb', 'xgb', 'cat', 'cat_allcat', 'realmlp'):
        add('kg_' + m, 'encodings notebook', o[m].values, t[m].values)

def l_yekenot():
    d = os.path.dirname(find('oof_preds.csv', 'realmlp-pytabkit')[0])
    add('yk_realmlp', 'yekenot', by_id(f'{d}/oof_preds.csv', tr['id'], 'satisfaction'),
        by_id(f'{d}/submission.csv', te['id'], 'satisfaction'))

def l_hermengardo():
    d = os.path.dirname(find('oof_train.npy', 'single-catboost-eda')[0])
    add('hm_cat', 'hermengardo', np.load(f'{d}/oof_train.npy'), np.load(f'{d}/oof_test.npy'))

def l_dmitriy():
    d = os.path.dirname(find('oof_cb_cat.npy', 'catboost-all-categorical')[0])
    add('dk_cbcat', 'dmitriykolmikov', np.load(f'{d}/oof_cb_cat.npy'), np.load(f'{d}/test_cb_cat.npy'))

def l_amanatar_v5():
    z = np.load(find('oof_artifacts.npz', 'original-prior-stack-v5')[0])
    assert (z['y'] == y).all()
    for m in ('lgbm_champion', 'lgbm_te', 'lgbm_te_rows', 'xgb_gpu', 'catboost_gpu'):   # base members only
        add('av5_' + m, 'amanatar v5', z['oof_' + m], z['test_' + m])

def l_goodpjw_extra():
    d = os.path.dirname(find('members.csv', 'goodpjw2008-extra-members')[0])
    for m in pd.read_csv(f'{d}/members.csv')['member']:
        if 'stack' in m:                  # the dataset also carries a frozen stack: a stack of stacks inflates nested CV
            continue
        add('gpx_' + m, 'goodpjw2008 extra', np.load(f'{d}/oof_{m}.npy'), np.load(f'{d}/test_{m}.npy'))

def l_ablation_members():                                                   # the new members
    d = os.path.dirname(find('members.csv', 'ablation-members-oof')[0])
    for m in pd.read_csv(f'{d}/members.csv')['member']:
        add('kx_' + m, OURS, np.load(f'{d}/oof_{m}.npy'), np.load(f'{d}/test_{m}.npy'))


def l_candidates():                                                         # screened in section 2b
    # Found by their files, not their names (a notebook's output folder keeps the name of its first save):
    # every attached folder that holds members.csv next to oof_allcat_*.npy or oof_pfn_*.npy files.
    hits = sorted(p for p in glob.glob(f'{INPUT}/**/members.csv', recursive=True)
                  if glob.glob(os.path.join(os.path.dirname(p), 'oof_allcat_*.npy'))
                  or glob.glob(os.path.join(os.path.dirname(p), 'oof_pfn_*.npy')))
    assert hits, 'not attached: a candidate notebook (all-categorical CatBoost variants or TabPFN members)'
    for p in hits:
        d = os.path.dirname(p)
        print(f'  candidates from {d}')
        for m in pd.read_csv(p)['member']:
            add('cx_' + m, CAND, np.load(f'{d}/oof_{m}.npy'), np.load(f'{d}/test_{m}.npy'))


LOADERS = [l_cdeotte124, l_cdeotte_tfm, l_golem, l_sadamtorres, l_forge, l_abdullah, l_koumeimaki, l_sachith7,
           l_kratosyan, l_encodings_notebook, l_yekenot, l_hermengardo, l_dmitriy, l_amanatar_v5,
           l_goodpjw_extra, l_ablation_members, l_candidates]          # ours last: a copy of a public member is then dropped
skipped = []
for fn in LOADERS:
    n0 = len(names)
    try:
        fn()
        print(f'{fn.__name__:24s} +{len(names) - n0}')
    except Exception as e:                                # a missing or changed source is skipped, not fatal
        del names[n0:], source[n0:], cols[n0:], tcols[n0:]
        skipped.append(fn.__name__)
        print(f'{fn.__name__:24s} skipped ({type(e).__name__}: {str(e)[:90]})')
print(f'\n{len(names)} members loaded in {time.time() - T0:.0f}s | skipped: {skipped or "none"}')

In [ ]:
# Drop exact copies (the same predictions published twice); the first occurrence is kept
Z, ZT = np.column_stack(cols), np.column_stack(tcols)
del cols, tcols
C = np.corrcoef(Z.T)
dup = [j for j in range(1, len(names)) if (C[:j, j] > 0.999999).any()]
keep = [j for j in range(len(names)) if j not in set(dup)]
print(f'dropped {len(dup)} exact copies:', [names[j] for j in dup][:20], '...' if len(dup) > 20 else '')
Z, ZT = Z[:, keep], ZT[:, keep]
names, source = [names[j] for j in keep], [source[j] for j in keep]
del C

auc = pd.Series([roc_auc_score(y, Z[:, i]) for i in range(len(names))], index=names)
by_src = pd.DataFrame({'source': source, 'auc': auc.values}).groupby('source')['auc'].agg(members='size', best='max')
display(by_src.sort_values('best', ascending=False).round(6))
print(f'{Z.shape[1]} members x {Z.shape[0]:,} OOF rows')

## 2. Nested cross-validation

`LogisticRegression(C=0.003)` on the logits: with more than 200 highly correlated inputs a strong L2 penalty works
best (it is the value the public stacks converged on). Each row's stack prediction comes from a stacker fitted on
the other four meta folds, so the score below carries no optimism from the weights. Three stacks are compared:
the public pool alone, the public pool plus the 17 new members, and the 17 new members on their own.

In [ ]:
META = list(StratifiedKFold(5, shuffle=True, random_state=7).split(Z, y))


def _fold(Zm, a, b, C):
    return b, LogisticRegression(C=C, max_iter=5000).fit(Zm[a], y[a]).decision_function(Zm[b])


def nested(idx, C=0.003):
    Zm, oof = Z[:, idx], np.zeros(len(y))
    for b, v in Parallel(n_jobs=min(5, os.cpu_count()))(delayed(_fold)(Zm, a, b, C) for a, b in META):
        oof[b] = v
    return roc_auc_score(y, oof), [roc_auc_score(y[b], oof[b]) for _, b in META]


public = [i for i, s in enumerate(source) if s not in (OURS, CAND)]
ours = [i for i, s in enumerate(source) if s == OURS]
cands = [i for i, s in enumerate(source) if s == CAND]
base = public + ours
t0 = time.time()
LADDER = {}
for label, idx, C_ in [('public pool', public, 0.003), ('public pool + 17 new', base, 0.003),
                       ('17 new members alone', ours, 1.0)]:
    if not idx:
        continue
    LADDER[label] = nested(idx, C_)
    s, fs = LADDER[label]
    print(f'{label:24s} {len(idx):4d} members   nested CV {s:.6f}   folds ' + ' '.join(f'{v:.5f}' for v in fs))

if 'public pool' in LADDER and 'public pool + 17 new' in LADDER:
    (s1, f1), (s2, f2) = LADDER['public pool'], LADDER['public pool + 17 new']
    print(f'\nthe 17 new members add {s2 - s1:+.6f}, {sum(b > a for a, b in zip(f1, f2))} of 5 folds up')
print(f'({time.time() - t0:.0f}s)')

**Result.** All 16 sources loaded (245 members), and one exact copy was dropped (`gpx_lgb_v3_k`, identical to a
member of the 124), leaving 244.

| stack | members | nested CV | per fold |
|---|---|---|---|
| public pool | 227 | 0.962205 | 0.96285 0.96173 0.96192 0.96213 0.96239 |
| **public pool + 17 new** | **244** | **0.962224** | 0.96289 0.96175 0.96192 0.96214 0.96242 |
| 17 new members alone (C = 1) | 17 | 0.961540 | 0.96233 0.96109 0.96124 0.96136 0.96169 |

- **The public pool reproduces the best public stack:** 0.962205 is exactly the nested CV goodpjw2008 reports for
  their 228 members, which is a check that the loader and the folds are right.
- **The 17 new members add +0.000020, and on every fold**, which is about what goodpjw2008's own unpublished members
  added to the same pool. Members nobody has seen are the one thing left that a pool of 227 does not already contain.
- **On their own the 17 stack to 0.961540**, above the 0.961496 that a rank blend of five of them reached in the
  development notebook: the older and weaker members still add something once a stacker can weigh them.

## 2b. Screen the candidates

Each candidate is added on its own to the stack above (public pool + 17) and kept only if nested CV rises by at least
`MIN_GAIN` (0.000005, a quarter of what the 17 new members add) **and** at least 4 of the 5 folds improve, so a
candidate that looks good by chance on one fold is not adopted. The kept ones are then measured together, and the
final stack uses them only if they still clear `MIN_GAIN` as a group. Smaller gains are below what nested CV can tell
apart from noise, and the public LB shows only five decimals.

In [ ]:
MIN_GAIN = 0.000005
KEEP = []
if cands:
    s_base, f_base = LADDER['public pool + 17 new']
    rows = []
    for i in cands:
        s_, fs = nested(base + [i])
        up = sum(b > a for a, b in zip(f_base, fs))
        rows.append({'candidate': names[i], 'alone OOF AUC': auc[names[i]], 'delta': s_ - s_base, 'folds up': up})
        if s_ - s_base >= MIN_GAIN and up >= 4:
            KEEP.append(i)
    display(pd.DataFrame(rows).set_index('candidate').round(6))
    if KEEP:
        s_k, f_k = nested(base + KEEP)
        print(f'base + {len(KEEP)} kept candidates: nested CV {s_k:.6f} ({s_k - s_base:+.6f}, '
              f'{sum(b > a for a, b in zip(f_base, f_k))} of 5 folds up)')
        if s_k - s_base < MIN_GAIN:
            KEEP = []
print('kept:', [names[i] for i in KEEP] or 'none')
FINAL = base + KEEP

## 3. Fit on all rows and write the submission

`submission.csv` is the stack of the public pool, the 17 new members and any candidates kept in section 2b.
`submission_public_pool.csv` (without the 17 and the candidates) is written alongside for comparison.

In [ ]:
def fit_and_write(idx, path):
    stack = LogisticRegression(C=0.003, max_iter=5000).fit(Z[:, idx], y)
    z = stack.decision_function(ZT[:, idx])
    pd.DataFrame({'id': te['id'].values, 'satisfaction': 1 / (1 + np.exp(-z))}).to_csv(path, index=False)
    return pd.Series(stack.coef_[0], index=[names[i] for i in idx])


w = fit_and_write(FINAL, 'submission.csv')
if ours:
    fit_and_write(public, 'submission_public_pool.csv')
print('largest |weights|:')
print(w.reindex(w.abs().sort_values(ascending=False).index).head(15).round(3).to_string())
print('\nweights of the new members:')
print(w[[n for n in w.index if n.startswith(('kx_', 'cx_'))]].sort_values(key=abs, ascending=False).round(3).to_string())
print(f'\ntotal {time.time() - T0:.0f}s')

**What the weights say.** The largest weights go to the tabular foundation models (TabFM 0.225, the 15-fold
TabPFN-3.5 0.206, LimiX-2 0.176): the clearest source of diversity in this pool. Of the 17 new members, the two
**all-categorical CatBoosts** carry the weight: `kx_cat_allcat_v1_k5` (0.108) has the 5th-largest absolute weight of
all 244 members, `kx_cat_allcat_v2_k5` (0.077) is in the top 15. The weaker of the two (0.961095 against 0.961127
alone) gets the larger weight, so its value is in ranking passengers differently, not in its own score. Negative
weights are normal in a stack of this many correlated members: the regression uses some members to subtract what
they share with others.

## 4. A twin with the best public stack (extra file, not validated)

The stacks at the top of the public LB are built on nearly the same public pool as this one, each with a few
members the others lack. `submission_twin.csv` averages the percentile ranks of this stack and one attached public
stack output, half and half. The weight is fixed in advance, not tuned on the leaderboard, but a test-only file has
no OOF, so this blend cannot be cross-validated: it is a hedge for the second final submission, and `submission.csv`
stays the CV-chosen stack.

Attach one of these (the first found is used): any dataset holding `0.96188.csv` (ewtherka's family-wise L2 stack,
republished by nina2025 as `ps-s6e10-02`), ewtherka's notebook itself, or dariushafshar's notebook
`s6e10-247-public-oofs-one-stack-lb-0-96187` (LB 0.96187).

In [ ]:
TWIN = [('0.96188.csv', ''), ('submission.csv', 'l2-family-wise'), ('submission.csv', '247-public-oofs')]
mine = pd.read_csv('submission.csv').set_index('id').loc[te['id'], 'satisfaction']
for pattern, hint in TWIN:
    hits = [f for f in glob.glob(f'{INPUT}/**/{pattern}', recursive=True) if hint in f]
    if not hits:
        continue
    other = pd.read_csv(hits[0]).set_index('id').loc[te['id'], 'satisfaction']
    assert other.notna().all() and len(other) == len(te), hits[0]
    twin = 0.5 * mine.rank(pct=True).values + 0.5 * other.rank(pct=True).values
    pd.DataFrame({'id': te['id'].values, 'satisfaction': twin}).to_csv('submission_twin.csv', index=False)
    print(f'twin with {hits[0]}\nrank correlation with this stack: {mine.corr(other, method="spearman"):.6f}')
    break
else:
    print('no public stack attached: submission_twin.csv not written')

## Credits

This notebook is a combiner: almost all of its value is other people's published models. **If it helps you, please
upvote the sources**, not only this page.

- **Chris Deotte**: the 124-member public OOF collection (a frozen copy of
  [kratosyan's library](https://www.kaggle.com/code/kratosyan/s6e10-stacking-every-public-oof-library), which holds
  members by goodpjw2008, sachith7, megayak, mitudru, busyaprime, arhancanli12, thisray, samanyu1808, hemingweb,
  wangxintong111, najiama and others) and the tabular-foundation-model predictions, plus the logit pass-through
  convention.
- **goodpjw2008**: the [228-member meta stack](https://www.kaggle.com/code/goodpjw2008/s6e10-228-member-meta-stack-lb-0-96184)
  whose loader this one is adapted from, the extra-members dataset, and the measurements behind `C = 0.003`.
- **yeashusemwal**: [Secrets to satisfaction: it's stacking](https://www.kaggle.com/code/yeashusemwal/secrets-to-satisfaction-it-s-stacking)
  (strong L2, meta folds on a different seed).
- **Member authors**: dariushafshar, sadamtorres, amanatar, abdullahsafwan333, koumeimaki, sachith7, kratosyan,
  Vladimir Demidov (yekenot), Hermengardo, dmitriykolmikov.
- **TabPFN-3.5** members come with Prior Labs' non-commercial licence; their authors note that it allows use in
  Kaggle competitions.